# Capa 1: Ingesta y Trazabilidad Bronze (Raw Data)
**Objetivo:** Cargar el archivo original desde `data/landing/`, normalizar encabezados y guardar el dataset crudo en formato CSV en `data/bronze/` junto con metadatos de auditoría.

Configuración de Rutas Portátiles

In [1]:
import os
from datetime import datetime
from pathlib import Path
import pandas as pd

# Detectar carpeta raíz del proyecto (funciona en cualquier PC)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_LANDING = BASE_DIR / "data" / "landing"
DIR_BRONZE = BASE_DIR / "data" / "bronze"

# Crear carpetas de destino si no existen
DIR_BRONZE.mkdir(parents=True, exist_ok=True)

print(f"Ruta Base del Proyecto: {BASE_DIR}")
print(f"Directorio Landing:    {DIR_LANDING}")
print(f"Directorio Bronze:     {DIR_BRONZE}")

Ruta Base del Proyecto: c:\Users\DRODRIGUEZQU\Downloads\proyecto_dulcinea_vih_mejorado\proyecto_dulcinea_vih
Directorio Landing:    c:\Users\DRODRIGUEZQU\Downloads\proyecto_dulcinea_vih_mejorado\proyecto_dulcinea_vih\data\landing
Directorio Bronze:     c:\Users\DRODRIGUEZQU\Downloads\proyecto_dulcinea_vih_mejorado\proyecto_dulcinea_vih\data\bronze


Carga desde Landing y Validación Inicial

In [2]:
# Localizar archivo Excel en la carpeta landing
archivos_excel = list(DIR_LANDING.glob("*.xlsx"))

if not archivos_excel:
    raise FileNotFoundError(
        "No se encontró ningún archivo .xlsx en 'data/landing/'."
    )

ruta_excel = archivos_excel[0]
df_raw = pd.read_excel(ruta_excel)

# Validación Celda 3
print(f"Archivo cargado: {ruta_excel.name}")
print(f"Total de registros leídos: {len(df_raw):,}")
print(f"Total de columnas leídas:  {len(df_raw.columns)}")

Archivo cargado: Base_Datos_Simulada_Cohorte_VIH_800_Registros.xlsx
Total de registros leídos: 850
Total de columnas leídas:  117


Normalización de Encabezados y Metadatos

In [3]:
df_bronze = df_raw.copy()

# Limpiar saltos de línea (\n), espacios dobles y espacios en extremos de las columnas
df_bronze.columns = [
    " ".join(str(col).replace("\n", " ").split()) for col in df_bronze.columns
]

# Agregar metadatos de trazabilidad
df_bronze["_fecha_ingesta_bronze"] = datetime.now()
df_bronze["_fuente_archivo"] = ruta_excel.name

# Validación Celda 4
print("Encabezados normalizados. Primeras 5 columnas:")
print(list(df_bronze.columns[:5]))

Encabezados normalizados. Primeras 5 columnas:
['Llave', 'TD', 'Numero de Identificacion', 'Primer Apellido', 'Segundo Apellido']


Exportación a CSV Bronze y Validación Final

In [4]:
ruta_csv_bronze = DIR_BRONZE / "cohorte_vih_bronze.csv"

try:
    df_bronze.to_csv(ruta_csv_bronze, index=False, encoding="utf-8-sig")

    # Validación Celda 5
    assert (
        ruta_csv_bronze.exists()
    ), "Error: El archivo CSV Bronze no fue generado."
    print(f" ¡Éxito! Archivo guardado correctamente en: {ruta_csv_bronze}")
    print(
        f" Filas procesadas: {len(df_bronze):,} | Columnas: {len(df_bronze.columns)}"
    )

except PermissionError:
    print(
        "\n ⚠️ ERROR DE PERMISOS (PermissionError):"
        "\n El archivo 'cohorte_vih_bronze.csv' está siendo utilizado por otro programa (Excel, Power BI, etc.)."
        "\n -> Cierra el archivo en Excel y vuelve a ejecutar esta celda."
    )

 ¡Éxito! Archivo guardado correctamente en: c:\Users\DRODRIGUEZQU\Downloads\proyecto_dulcinea_vih_mejorado\proyecto_dulcinea_vih\data\bronze\cohorte_vih_bronze.csv
 Filas procesadas: 850 | Columnas: 119


## Diccionario de Variables - Proyecto DULCINEA (117 Variables)

El conjunto de datos de la cohorte VIH del Departamento de Antioquia consta de **117 variables** agrupadas en 6 módulos funcionales para el análisis epidemiológico y la gestión del abandono:

| Módulo Temático | Cant. | Variables Clave | Uso Operativo en DULCINEA |
| :--- | :--- | :--- | :--- |
| **1. Demografía y Ubicación** | 23 | `TD`, `Numero de Identificacion`, `Nombre_Completo`, `Edad`, `Sexo`, `Municipio residencia`, `Subregion` | Identificación única de pacientes, análisis demográfico y segmentación geográfica. |
| **2. Sistema de Salud y Aseguramiento** | 22 | `Regimen de afiliacion`, `Estado afiliacion EAPB`, `IPS primaria`, `Reporte en SIVIGILA`, `Reportado en CAC` | Articulación de fuentes (SIVIGILA vs CAC) y trazabilidad del aseguramiento. |
| **3. Manejo Clínico VIH y TAR** | 21 | `36. Fecha del diagnostico de VIH`, `Estadio Clinico`, `CD4`, `Carga viral`, `Recibe TAR`, `Esquema TAR` | Identificación de falla virológica y cálculo de la variable principal de **Abandono**. |
| **4. Coinfecciones y Profilaxis** | 19 | `Hepatitis B/C`, `Tuberculosis (TB)`, `Tamizaje Sífilis`, `Profilaxis MAC / Criptococo / Pneumocystis` | Evaluación de comorbilidades epidemiológicas de alto impacto sanitario. |
| **5. Salud Materno-Perinatal** | 5 | `GESTANTE`, `Mujer Gestante`, `Edad gestacional`, `Fecha del parto`, `Resultado Gestación` | Seguimiento focalizado para la prevención de la transmisión vertical materna. |
| **6. Laboratorios y Cohortes de Riesgo** | 27 | `Colesterol`, `Creatinina`, `Glucemia`, `Riesgo Cardiovascular`, `Cohortes`, `Consultas` | Monitoreo metabólico integral y frecuencia de atención médica especializada. |